In [1]:
# ================================================================
# BLOQUE 1: IMPORTS GENERALES Y CONFIGURACIÓN
# ================================================================
import os
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler, random_split
from torchvision import datasets, models, transforms
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import time, copy
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# ================================================================
# BLOQUE 2: RUTAS A DATASETS
# ================================================================

trashnet_dir = r"D:\Documentos\IPN\6to Semestre\Metodologia_de_la_investigacion\Clasificacion_residuos\data_preprocessed\ResNet18\TrashNet"
taco_dir     = r"D:\Documentos\IPN\6to Semestre\Metodologia_de_la_investigacion\Clasificacion_residuos\data_preprocessed\ResNet18\TACO"

output_dir = r"D:\Documentos\IPN\6to Semestre\Metodologia_de_la_investigacion\Clasificacion_residuos\Programas\ResNet\Modelos_experimentos"
os.makedirs(output_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# ================================================================
# BLOQUE 3: FUNCIONES AUXILIARES
# ================================================================

def split_stratified(dataset, train_ratio=0.7, val_ratio=0.15, seed=42):
    np.random.seed(seed)
    targets = np.array(dataset.targets)
    classes = np.unique(targets)
    
    train_idx, val_idx, test_idx = [], [], []
    for c in classes:
        c_idx = np.where(targets == c)[0]
        np.random.shuffle(c_idx)
        n_total = len(c_idx)
        n_train = int(train_ratio * n_total)
        n_val = int(val_ratio * n_total)
        
        train_idx += list(c_idx[:n_train])
        val_idx   += list(c_idx[n_train:n_train+n_val])
        test_idx  += list(c_idx[n_train+n_val:])
    
    return train_idx, val_idx, test_idx


def make_balanced_sampler(subset, full_dataset):
    targets = np.array(full_dataset.targets)
    subset_targets = targets[subset.indices]
    class_counts = Counter(subset_targets)
    weights = {cls: 1.0/count for cls, count in class_counts.items()}
    sample_weights = [weights[label] for label in subset_targets]
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

In [4]:
# ================================================================
# BLOQUE 4: FUNCIÓN DE ENTRENAMIENTO CON HISTORIAL PARA CURVAS
# ================================================================

def train_model(model, criterion, optimizer, dataloaders, dataset_sizes,
                num_epochs=20, patience=4, save_name="modelo.pth"):

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    since = time.time()
    best_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    no_improve = 0

    for epoch in range(num_epochs):
        print(f"\nÉpoca {epoch+1}/{num_epochs}")
        print("-" * 50)

        for phase in ["train", "val"]:
            model.train() if phase=="train" else model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase=="train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase=="train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item()*inputs.size(0)
                running_corrects += torch.sum(preds==labels)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc  = running_corrects.double()/dataset_sizes[phase]

            print(f"{phase.upper()} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")

            # registrar historial
            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc.item())

            # early stopping
            if phase == "val":
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_wts = copy.deepcopy(model.state_dict())
                    no_improve = 0
                else:
                    no_improve += 1

        if no_improve >= patience:
            print("\nEARLY STOPPING activado")
            break

    model.load_state_dict(best_wts)
    torch.save(model.state_dict(), os.path.join(output_dir, save_name))

    print(f"\nMejor ACC validación: {best_acc:.4f}")
    return model, history

In [5]:
# ================================================================
# BLOQUE 5: FUNCIÓN PARA GRAFICAR CURVAS DE APRENDIZAJE
# ================================================================

def plot_curves(history, title="Curvas de aprendizaje"):
    plt.figure(figsize=(12,5))

    # Loss
    plt.subplot(1,2,1)
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Val Loss")
    plt.title(title + " - Loss")
    plt.xlabel("Épocas")
    plt.legend()

    # Accuracy
    plt.subplot(1,2,2)
    plt.plot(history["train_acc"], label="Train Acc")
    plt.plot(history["val_acc"], label="Val Acc")
    plt.title(title + " - Accuracy")
    plt.xlabel("Épocas")
    plt.legend()

    plt.tight_layout()
    plt.show()

In [6]:
# ================================================================
# FUNCIÓN DE EVALUACIÓN COMPLETA
# Matriz de confusión (gráfica) + metrics por defecto en consola
# ================================================================

def evaluar_en_test(model, loader, class_names, modelo_nombre="Modelo"):
    model.eval()
    all_preds, all_labels = [], []

    # 1. Recolectar predicciones reales vs predichas
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # ===========================================================
    # 2. Reporte de clasificación (texto)
    # ===========================================================
    print("\n===============================================")
    print(f"   REPORTES DE EVALUACIÓN - {modelo_nombre}")
    print("===============================================\n")

    print(classification_report(
        all_labels,
        all_preds,
        target_names=class_names,
        digits=3
    ))

    # ===========================================================
    # 3. Matriz de confusión (gráfica)
    # ===========================================================

    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(7,6))
    plt.imshow(cm, cmap='Blues')
    plt.title(f"Matriz de Confusión - {modelo_nombre}")
    plt.xlabel("Predicción")
    plt.ylabel("Etiqueta Real")
    plt.xticks(np.arange(len(class_names)), class_names, rotation=45)
    plt.yticks(np.arange(len(class_names)), class_names)

    # valores dentro de celdas
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=12)

    plt.colorbar()
    plt.tight_layout()
    plt.show()

    # ===========================================================
    # 4. Exactitud total
    # ===========================================================
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    print(f"Exactitud global en TEST ({modelo_nombre}): {acc:.4f}")

    return acc

In [7]:
# ================================================================
# BLOQUE 6: ENTRENAMIENTO TRASHNET - SIN DATA AUGMENTATION
# ================================================================

transform = transforms.ToTensor()

# --- AHORA SE CARGA TACO, PERO MANTENEMOS EL NOMBRE trashnet_full ---
taco_full = datasets.ImageFolder(taco_dir, transform=transform)
print("Clases TACO:", taco_full.classes)

# split estratificado
train_idx, val_idx, test_idx = split_stratified(taco_full)
train_ds = Subset(taco_full, train_idx)
val_ds   = Subset(taco_full, val_idx)
test_ds  = Subset(taco_full, test_idx)

sampler  = make_balanced_sampler(train_ds, taco_full)

batch = 32
train_loader = DataLoader(train_ds, batch_size=batch, sampler=sampler)
val_loader   = DataLoader(val_ds, batch_size=batch)
test_loader  = DataLoader(test_ds, batch_size=batch)

dataloaders = {"train": train_loader, "val": val_loader}
dataset_sizes = {"train": len(train_ds), "val": len(val_ds)}

# --- MODELO ---
model_taco = models.resnet18(weights="IMAGENET1K_V1")
for p in model_taco.parameters():
    p.requires_grad = True

# capa final con clases de TACO (aunque el nombre sea trashnet_full)
model_taco.fc = nn.Linear(model_taco.fc.in_features, len(taco_full.classes))
model_taco = model_taco.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_taco.fc.parameters(), lr=1e-3)

model_taco, hist_taco = train_model(
    model_taco, criterion, optimizer,
    dataloaders, dataset_sizes,
    num_epochs=10, patience=4,
    save_name="resnet18_taco_pretrain.pth"
)

plot_curves(hist_taco, "TACO - Sin Data Augmentation")

Clases TACO: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

Época 1/10
--------------------------------------------------
TRAIN | Loss: 1.5277 | Acc: 0.3957
VAL | Loss: 1.5085 | Acc: 0.4211

Época 2/10
--------------------------------------------------
TRAIN | Loss: 1.1944 | Acc: 0.5765
VAL | Loss: 1.3530 | Acc: 0.4916

Época 3/10
--------------------------------------------------
TRAIN | Loss: 1.0821 | Acc: 0.6131
VAL | Loss: 1.2328 | Acc: 0.5452

Época 4/10
--------------------------------------------------
TRAIN | Loss: 1.0046 | Acc: 0.6521
VAL | Loss: 1.2488 | Acc: 0.5299

Época 5/10
--------------------------------------------------
TRAIN | Loss: 1.0005 | Acc: 0.6491
VAL | Loss: 1.2946 | Acc: 0.5253

Época 6/10
--------------------------------------------------
TRAIN | Loss: 0.9264 | Acc: 0.6848
VAL | Loss: 1.3130 | Acc: 0.5100

Época 7/10
--------------------------------------------------


KeyboardInterrupt: 

In [ ]:
# ================================================================
# BLOQUE 7: ENTRENAMIENTO TACO - CON DATA AUGMENTATION
# ================================================================

train_transform_taco = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2,0.2),
    transforms.ToTensor()
])

base_transform_taco = transforms.ToTensor()

# --- AHORA ESTE BLOQUE USA TACO EN LUGAR DE TRASHNET ---
taco_aug_full = datasets.ImageFolder(taco_dir, transform=train_transform_taco)
print("Clases TACO:", taco_aug_full.classes)

# split reproducible
train_size = int(0.7 * len(taco_aug_full))
val_size   = int(0.15 * len(taco_aug_full))
test_size  = len(taco_aug_full) - train_size - val_size

torch.manual_seed(42)
train_ds_aug, val_ds_aug, test_ds_aug = random_split(
    taco_aug_full, [train_size, val_size, test_size]
)

# para validación y test: sin augmentación
val_ds_aug.dataset.transform  = base_transform_taco
test_ds_aug.dataset.transform = base_transform_taco

train_loader_aug = DataLoader(train_ds_aug, batch_size=32, shuffle=True)
val_loader_aug   = DataLoader(val_ds_aug, batch_size=32)
test_loader_aug  = DataLoader(test_ds_aug, batch_size=32)

dataloaders_aug = {"train": train_loader_aug, "val": val_loader_aug}
dataset_sizes_aug = {"train": len(train_ds_aug), "val": len(val_ds_aug)}

# Modelo
model_taco_aug = models.resnet18(weights="IMAGENET1K_V1")
for p in model_taco_aug.parameters():
    p.requires_grad = True

# número de clases de TACO (aunque el nombre sea trashnet_aug_full)
model_taco_aug.fc = nn.Linear(model_taco_aug.fc.in_features, len(taco_aug_full.classes))
model_taco_aug = model_taco_aug.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_taco_aug.fc.parameters(), lr=1e-3)

model_taco_aug, hist_taco_aug = train_model(
    model_taco_aug, criterion, optimizer,
    dataloaders_aug, dataset_sizes_aug,
    num_epochs=10, patience=4,
    save_name="resnet18_taco_aug_pretrain.pth"
)

plot_curves(hist_taco_aug, "TACO - Con Data Augmentation")

In [ ]:
# ================================================================
# BLOQUE 8: EVALUACIÓN TACO (CON MATRICES DE CONFUSIÓN)
# ================================================================

print("\n=== EVALUACIÓN TACO - SIN DA ===")
acc_taco = evaluar_en_test(
    model_taco,
    test_loader,
    taco_full.classes,
    modelo_nombre="TACO - Sin DA"
)

print("\n=== EVALUACIÓN TACO - CON DA ===")
acc_taco_aug = evaluar_en_test(
    model_taco_aug,
    test_loader_aug,
    taco_aug_full.classes,
    modelo_nombre="TACO - Con DA"
)

print("\n>> Exactitud TACO Sin DA:", acc_taco)
print(">> Exactitud TACO Con DA:", acc_taco_aug)

In [ ]:
# ================================================================
# BLOQUE 9: Carga de TrashNet como conjunto de prueba
# ================================================================
print("\nCargando TrashNet (mismas 6 clases que TrashNet)...")

transform_tn = transforms.ToTensor()

# Se asume que TACO ya fue filtrado a las mismas 6 clases
tn_full = datasets.ImageFolder(trashnet_dir, transform=transform_tn)

print("Clases detectadas en TrashNet:", tn_full.classes)

# Puedes usar TODO TACO como test, o dividir (pero no entrenaremos nada)
test_loader_tn = DataLoader(tn_full, batch_size=32, shuffle=False)

In [ ]:
# ================================================================
# BLOQUE 10: Evaluación del modelo sin DA
# ================================================================
print("\n=== Evaluación de TrashNet SIN DA sobre TACO ===")

model_taco.eval()

acc_tn_sinDA =  evaluar_en_test(
    model_taco,
    test_loader_tn,
    tn_full.classes,
    modelo_nombre="TACO SIN DA → TrashNet"
)

print("\nAccuracy TrashNet usando modelo TACO SIN DA:", acc_tn_sinDA)


In [ ]:
# ================================================================
# BLOQUE 11: Evaluación del modelo con DA
# ================================================================
print("\n=== Evaluación de TrashNet CON DA sobre TACO ===")

model_taco_aug.eval()

acc_tn_conDA =  evaluar_en_test(
    model_taco_aug,
    test_loader_tn,
    tn_full.classes,
    modelo_nombre="TACO con DA → TrashNet"
)

print("\nAccuracy TrashNet usando modelo TACO CON DA:", acc_tn_conDA)


In [ ]:
# ================================================================
# BLOQUE 10: Guardado de ambos modelos
# ================================================================
torch.save(model_taco.state_dict(),
           os.path.join(output_dir, "resnet18_tacoS3_sinDA_final.pth"))

torch.save(model_taco_aug.state_dict(),
           os.path.join(output_dir, "resnet18_tacoS3_conDA_final.pth"))

print("\nModelos guardados correctamente.")